# Regularized Regression — Notebook 05 · Logistic Regression

**Regularized Regression — From Likelihood to Lasso**

Covers **Task 6**: a Bernoulli likelihood in place of a Gaussian one, the same ℓ₁ and ℓ₂ priors, ROC and the threshold as a policy choice, and a measurement of what dichotomising the target cost.

---

### How to run this

Put `uber_surge_mumbai.csv` (and, for notebook 02, `new_city_features.csv` and
`new_city_truth.csv`) in a `data/` folder beside this notebook, or upload them to your
Colab session. If the files are missing, the next cell regenerates them from the same
seed, so nothing here breaks either way.

In [ ]:
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True, linewidth=110)
pd.set_option('display.width', 120, 'display.max_columns', 30)

# --- a consistent look for every figure in this course -----------------------
plt.rcParams.update({
    'figure.figsize': (9, 4.6), 'figure.dpi': 110,
    'axes.grid': True, 'grid.alpha': .28, 'grid.linewidth': .7,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c9cbc3', 'axes.labelcolor': '#4c514f',
    'axes.titlesize': 12, 'axes.titleweight': '600', 'axes.labelsize': 10,
    'xtick.color': '#767d7a', 'ytick.color': '#767d7a',
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.frameon': False, 'legend.fontsize': 9,
    'font.size': 10, 'figure.facecolor': 'white', 'axes.facecolor': 'white',
})
C = {'ols': '#767d7a', 'ridge': '#2b5fa8', 'lasso': '#b4700c',
     'enet': '#5b4bb5', 'accent': '#1f6f5c', 'rose': '#b03a52'}

# Look in the usual places: beside the notebook, in ./data, or one level up
# (which is where the repo keeps them if you opened this from notebooks/).
SEARCH = ['data', '.', '../data', '..', '../../data']

def load(name):
    """Load a course CSV, regenerating it from the seed if it cannot be found."""
    for d in SEARCH:
        path = os.path.join(d, name)
        if os.path.exists(path):
            return pd.read_csv(path)
    for cand in ('scripts/make_data.py', 'make_data.py', '../scripts/make_data.py'):
        if os.path.exists(cand):
            subprocess.run([sys.executable, cand], check=True)
            return load(name)
    raise FileNotFoundError(
        f"{name} not found. Download it from the course site's Data page and put it "
        f"in a data/ folder beside this notebook.")

df = load('uber_surge_mumbai.csv')
print(f'{len(df)} trips, {df.shape[1]} columns')
df.head(3)

In [ ]:
# ---------------------------------------------------------------------------
# Conventions used throughout this course. Read these once; they save hours.
#
#   Objective (identical to sklearn's ElasticNet, including the 1/2n):
#       (1/2n)||y - Xw||^2  +  alpha*rho*||w||_1  +  (alpha*(1-rho)/2)*||w||^2
#
#   So the Ridge CLOSED FORM needs  n*alpha  where sklearn's Ridge takes alpha:
#       w = (X^T X + n*alpha*I)^-1 X^T y
#
#   Standardize on the TRAINING rows only, then apply that scaler to the test
#   rows. Split by time, never shuffled: trips 1-150 train, 151-200 test.
# ---------------------------------------------------------------------------
BASE8 = ['is_peak', 'is_rain', 'traffic_speed_kmph', 'drivers_available_500m',
         'is_event_nearby', 'is_airport_pickup', 'is_weekend', 'open_requests_500m']
SHORT = {'is_peak': 'is_peak', 'is_rain': 'is_rain',
         'traffic_speed_kmph': 'traffic_speed', 'drivers_available_500m': 'drivers_avail',
         'is_event_nearby': 'is_event', 'is_airport_pickup': 'is_airport',
         'is_weekend': 'is_weekend', 'open_requests_500m': 'open_requests',
         'is_bad_weather': 'is_bad_weather'}
N_TRAIN, SIGMA, TAU = 150, 10.0, 1.0

def split(frame, features):
    X = frame[features].to_numpy(float)
    y = frame['surge_additive_inr'].to_numpy(float)
    return X[:N_TRAIN], X[N_TRAIN:], y[:N_TRAIN], y[N_TRAIN:]

def standardize(Xtr, Xte):
    mu, sd = Xtr.mean(0), Xtr.std(0)
    sd = np.where(sd == 0, 1e-8, sd)
    return (Xtr - mu) / sd, (Xte - mu) / sd, mu, sd

def rmse(y, yhat):
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def r2(y, yhat):
    return float(1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2))

print('conventions loaded')

## 6a — The label, and why least squares will not do

There is no binary column in the data, so we build one:
`high_surge = surge_additive_inr > 100`. That is a real operational trigger, and also a
deliberate loss of information. Section 6e measures the cost.

In [ ]:
CUT = 100.0
# TODO: build the binary label, then report positives in train/test and the base rate.
# TODO: your code here
raise NotImplementedError

In [ ]:
# TODO: fit OLS to the 0/1 label and count how many fitted values fall outside [0, 1].
# TODO: your code here
raise NotImplementedError

> **Write-up prompt (6a step 3).** The Bernoulli log-likelihood is
> $\sum_i [y_i \log p_i + (1-y_i)\log(1-p_i)]$ with $p_i = \sigma(x_i^\top w + b)$.
> Maximising it is minimising the average of its negative, which is the log-loss. Same
> manoeuvre as Task 1 with a different density: the loss comes from the noise model.

## 6b — Implement penalized logistic regression

With the intercept **unpenalized**:

$$\frac{1}{n}\sum_i \big[-y_i\log p_i-(1-y_i)\log(1-p_i)\big]
 + \alpha\rho\lVert w\rVert_1 + \frac{\alpha(1-\rho)}{2}\lVert w\rVert_2^2$$

There is no closed form. This is proximal gradient descent with FISTA acceleration: a
gradient step on the smooth part, then the same soft-threshold as Task 3.

In [ ]:
def sigmoid(z):
    pass

def log_loss(y, p):
    pass

def logistic_fit(Z, y, alpha=0.0, l1_ratio=0.0, max_iter=4000, tol=1e-10):
    # Minimise mean log-loss + alpha*rho*|w|_1 + alpha*(1-rho)/2*|w|^2.
    #
    # Hints:
    #   smooth gradient : Z.T @ (sigmoid(Zw+b) - y)/n + alpha*(1-rho)*w
    #   intercept       : mean(sigmoid(Zw+b) - y)   -- and do NOT penalize b
    #   step size       : 1/L with L = 0.25*max_i(||z_i||^2 + 1) + alpha*(1-rho)
    #   after the step  : soft-threshold w by step*alpha*rho
    # TODO: your code here
    raise NotImplementedError

### Verify against scikit-learn

With this objective `LogisticRegression` matches at $C = 1/(n\alpha)$.

> **Read this before you spend an hour debugging.** `solver='liblinear'` **penalizes the
> intercept** and our objective does not, so it disagrees by far more than solver
> tolerance and no tuning will close the gap. Use `lbfgs` for ℓ₂ or `saga` for ℓ₁ and
> elastic net. Setting `intercept_scaling` very large makes liblinear agree, which is how
> you confirm that is the cause.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.linear_model import LogisticRegression

n = len(ytr)
print(f"{'alpha':>7} {'rho':>5} {'max|w diff|':>13} {'|b diff|':>10}  solver")
for alpha, rho in [(0.01, 0.0), (0.1, 0.0), (0.02, 1.0), (0.05, 0.5)]:
    mine_w, mine_b, _ = logistic_fit(Ztr, ytr, alpha, rho, max_iter=80000, tol=1e-14)
    Cpar = 1.0 / (n * alpha)
    if rho == 0.0:
        sk = LogisticRegression(C=Cpar, solver='lbfgs', max_iter=50000, tol=1e-12)
        name = 'lbfgs'
    else:
        sk = LogisticRegression(C=Cpar, penalty='elasticnet', l1_ratio=rho,
                                solver='saga', max_iter=500000, tol=1e-12)
        name = 'saga'
    sk.fit(Ztr, ytr)
    dw = float(np.max(np.abs(mine_w - sk.coef_[0])))
    db = float(abs(mine_b - sk.intercept_[0]))
    print(f'{alpha:>7} {rho:>5} {dw:>13.2e} {db:>10.2e}  {name}')

lib = LogisticRegression(C=1/(n*0.02), penalty='l1', solver='liblinear',
                         max_iter=50000, tol=1e-12).fit(Ztr, ytr)
lib2 = LogisticRegression(C=1/(n*0.02), penalty='l1', solver='liblinear',
                          intercept_scaling=1e4, max_iter=50000, tol=1e-12).fit(Ztr, ytr)
mine_w, mine_b, _ = logistic_fit(Ztr, ytr, 0.02, 1.0, max_iter=80000, tol=1e-14)
print()
print(f'liblinear default      b = {lib.intercept_[0]:+.4f}   <- intercept penalized')
print(f'liblinear scaling=1e4  b = {lib2.intercept_[0]:+.4f}')
print(f'ours                   b = {mine_b:+.4f}   <- matches the rescaled version')

## 6c — Coefficient paths and the elimination order

In [ ]:
alphas = np.logspace(-3, 0.6, 40)
paths = {rho: np.array([logistic_fit(Ztr, ytr, a, rho)[0] for a in alphas])
         for rho in (0.0, 1.0)}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)
titles = ['$\\ell_2$ — shrinks, never reaches zero', '$\\ell_1$ — reaches zero exactly']
for ax, rho, title in zip(axes, (0.0, 1.0), titles):
    for j, f in enumerate(FEATS):
        ax.plot(alphas, paths[rho][:, j], lw=1.8, label=SHORT[f])
    ax.set_xscale('log'); ax.axhline(0, color='#c9cbc3', lw=1)
    ax.set(xlabel=r'$\alpha$ (log scale)', title=title)
axes[0].set_ylabel('coefficient (log-odds per s.d.)')
axes[1].legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.show()

order, seen = [], set()
for k, a in enumerate(alphas):
    for j in np.where(np.abs(paths[1.0][k]) < 1e-8)[0]:
        if j not in seen:
            seen.add(j); order.append((SHORT[FEATS[j]], round(float(a), 4)))
print('l1 elimination order (feature, alpha at which it died):')
for f, a in order:
    print(f'  {f:<16} {a}')

> **Write-up prompt (6c step 2).** Compare this with your Task 3c prediction. On this data
> the redundant weather features tend to go *before* `is_weekend`, even though
> `is_weekend` has no real effect at all — because `traffic_speed` already carries the
> weather information and ℓ₁ will not pay for it twice.
>
> So an ℓ₁ ordering ranks **marginal usefulness given the other features**, not true
> importance. That is the selection-instability caveat from Theory §5.3 in a new guise,
> and a good reason not to read a Lasso's survivors as a causal story.

In [ ]:
# TODO: cross-validate on LOG-LOSS (not accuracy) for no-penalty / l2 / l1 / elastic,
#       then report held-out log-loss, AUC, accuracy and non-zero count for each.
# TODO: your code here
raise NotImplementedError

## 6d — ROC, and choosing a threshold

In [ ]:
best = cls_table.loc[cls_table.test_logloss.idxmin()]
rho_best = {'no penalty': 0.0, 'l2': 0.0, 'l1': 1.0, 'elastic': 0.5}[best.model]
w_b, b_b, _ = logistic_fit(Ztr, ytr, best.alpha if best.alpha else 1e-6, rho_best)
p_te = sigmoid(Zte @ w_b + b_b)
print(f'best by CV log-loss: {best.model} (alpha={best.alpha})')


def confusion(y, p, t):
    hat = (p >= t).astype(int)
    return (int(((hat == 1) & (y == 1)).sum()), int(((hat == 1) & (y == 0)).sum()),
            int(((hat == 0) & (y == 0)).sum()), int(((hat == 0) & (y == 1)).sum()))


MISS_COST, FALSE_COST = 4, 1     # dispatch: a missed alert costs 4x a false alarm
print()
print(f"{'thresh':>7}{'TP':>5}{'FP':>5}{'TN':>5}{'FN':>5}{'recall':>9}{'precision':>11}{'cost':>8}")
best_t, best_cost = None, np.inf
for t in np.arange(0.05, 0.96, 0.05):
    tp, fp, tn, fn = confusion(yte, p_te, t)
    cost = MISS_COST * fn + FALSE_COST * fp
    if cost < best_cost:
        best_cost, best_t = cost, t
    rec = tp / (tp + fn) if tp + fn else float('nan')
    prec = tp / (tp + fp) if tp + fp else float('nan')
    print(f'{t:>7.2f}{tp:>5}{fp:>5}{tn:>5}{fn:>5}{rec:>9.3f}{prec:>11.3f}{cost:>8}')
print()
print(f'minimum expected cost at threshold {best_t:.2f}  (cost {best_cost})')
print('Note it is BELOW 0.5: when a miss costs 4x a false alarm, alert more readily.')

fpr, tpr = [], []
for t in np.r_[1.01, np.sort(p_te)[::-1], -0.01]:
    tp, fp, tn, fn = confusion(yte, p_te, t)
    tpr.append(tp / (tp + fn) if tp + fn else 0.0)
    fpr.append(fp / (fp + tn) if fp + tn else 0.0)

fig, ax = plt.subplots(figsize=(6.4, 5.6))
ax.plot(fpr, tpr, lw=2.6, color=C['accent'], label=f'ROC (AUC = {auc(yte, p_te):.3f})')
ax.plot([0, 1], [0, 1], '--', color=C['ols'], label='chance')
tp, fp, tn, fn = confusion(yte, p_te, best_t)
ax.plot(fp / (fp + tn), tp / (tp + fn), 'o', ms=10, color=C['rose'],
        label=f'cost-optimal t = {best_t:.2f}')
ax.set(xlabel='false positive rate', ylabel='true positive rate',
       title='Every point on the curve is a threshold you could choose')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

> **Write-up prompt (6d step 3).** The cost-minimising threshold is below 0.5, because a
> missed alert is four times as expensive as a false alarm. Note what did **not** change
> when you moved it: not a single coefficient. The threshold is a policy parameter and
> belongs beside the ₹150 cap from Task 5, not inside the model.

## 6e — Was the label worth making?

In [ ]:
# TODO: take your Ridge model from Task 2, predict rupees on the held-out trips,
#       threshold the prediction at CUT, and compare accuracy and AUC with your best
#       classifier. Then conclude.
# TODO: your code here
raise NotImplementedError

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
ok = yte == 1
ax.scatter(yhat_rupees[~ok], p_te[~ok], s=38, color=C['ridge'], alpha=.75, label='no alert')
ax.scatter(yhat_rupees[ok], p_te[ok], s=38, color=C['rose'], alpha=.75, label='alert')
ax.axvline(CUT, color=C['ols'], ls='--', lw=1.4)
ax.axhline(0.5, color=C['ols'], ls=':', lw=1.4)
ax.set(xlabel='Ridge prediction (Rs)', ylabel='classifier P(alert)',
       title='Two routes to the same yes/no — opposite corners are disagreements')
ax.legend()
plt.tight_layout(); plt.show()

dis = int(((yhat_rupees > CUT) != (p_te >= 0.5)).sum())
print(f'the two approaches disagree on {dis} of {len(yte)} held-out trips')

---

### Checklist

- [ ] Base rate quoted beside every accuracy
- [ ] OLS-on-0/1 shown leaving [0, 1], with a reason
- [ ] `logistic_fit` written from scratch, matching sklearn via **lbfgs / saga**
- [ ] The liblinear intercept trap identified
- [ ] Coefficient paths for ℓ₂ and ℓ₁; elimination order vs your Task 3c guess
- [ ] Model selection on **log-loss**, not accuracy
- [ ] ROC, AUC, and a threshold justified by the 4:1 cost ratio (**Deliverable 4**)
- [ ] 6e answered: did dichotomising the target cost anything?

Nothing in Task 6 needed a new idea about priors — only a new likelihood.